In [3]:
import requests
import pandas as pd
import os

API_KEY = "Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

ADDRESS = "0x846943093f519A47734765BEF9EE1136800bEb9C"

url = "https://api.etherscan.io/v2/api"

params = {
    "chainid": "11155111",
    "module": "account",
    "action": "txlist",
    "address": ADDRESS,
    "startblock": 0,
    "endblock": 999999999,
    "page": 1,
    "offset": 100,
    "sort": "asc",
    "apikey": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

if data["status"] == "1":

    df = pd.DataFrame(data["result"])

    # Create folder if it doesn't exist
    os.makedirs("../data", exist_ok=True)

    # Save CSV
    output_path = "../data/transactionA.csv"
    df.to_csv(output_path, index=False)

    print(f"Saved {len(df)} transactions to {output_path}")

else:
    print("API Error:", data)

Saved 57 transactions to ../data/transactionA.csv


### Inword Flow

In [5]:
import requests
import pandas as pd
import os
import time

# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = "Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

START_ADDRESS = "0x846943093f519A47734765BEF9EE1136800bEb9C"

MAX_HOPS = 10

URL = "https://api.etherscan.io/v2/api"

OUTPUT_PATH = "../data/transactionA_3hop.csv"

# Number of transactions returned per API request
OFFSET = 100

# Small delay between API requests
REQUEST_DELAY = 0.2


# ============================================================
# FUNCTION: GET TRANSACTIONS OF ONE ADDRESS
# ============================================================

def get_transactions(address):

    params = {
        "chainid": "11155111",       # Sepolia
        "module": "account",
        "action": "txlist",
        "address": address,
        "startblock": 0,
        "endblock": 999999999,
        "page": 1,
        "offset": OFFSET,
        "sort": "asc",
        "apikey": API_KEY
    }

    try:

        response = requests.get(
            URL,
            params=params,
            timeout=20
        )

        response.raise_for_status()

        data = response.json()

    except requests.exceptions.RequestException as e:

        print(f"Request failed for {address}")
        print(e)

        return []

    except ValueError:

        print(f"Invalid JSON response for {address}")

        return []


    # --------------------------------------------------------
    # Etherscan response handling
    # --------------------------------------------------------

    if data.get("status") == "1":

        return data.get("result", [])

    elif data.get("message") == "No transactions found":

        return []

    else:

        print(f"API error for {address}:")
        print(data)

        return []


# ============================================================
# MULTI-HOP COLLECTION
# ============================================================

all_transactions = []

visited_addresses = set()

current_addresses = {
    START_ADDRESS.lower()
}


# ============================================================
# HOP LOOP
# ============================================================

for hop in range(MAX_HOPS):

    print("\n" + "=" * 60)
    print(f"HOP {hop}")
    print("=" * 60)

    print(f"Addresses to investigate: {len(current_addresses)}")

    next_addresses = set()


    # --------------------------------------------------------
    # Investigate every address in this hop
    # --------------------------------------------------------

    for address in current_addresses:

        address = address.lower()

        # Don't request the same address twice
        if address in visited_addresses:
            continue

        print(f"\nFetching transactions for:")
        print(address)

        transactions = get_transactions(address)

        print(f"Transactions found: {len(transactions)}")

        visited_addresses.add(address)


        # ----------------------------------------------------
        # Process transactions
        # ----------------------------------------------------

        for tx in transactions:

            tx_from = tx.get("from")

            tx_to = tx.get("to")

            tx_hash = tx.get("hash")


            if tx_from:

                tx_from = tx_from.lower()

            if tx_to:

                tx_to = tx_to.lower()


            # ------------------------------------------------
            # Save transaction
            # ------------------------------------------------

            transaction = {

                "hop": hop,

                "hash": tx_hash,

                "blockNumber": tx.get("blockNumber"),

                "timeStamp": tx.get("timeStamp"),

                "from": tx_from,

                "to": tx_to,

                "value": tx.get("value"),

                "gasUsed": tx.get("gasUsed"),

                "gasPrice": tx.get("gasPrice"),

                "isError": tx.get("isError"),

                "tx_for_address": address
            }

            all_transactions.append(transaction)


            # ------------------------------------------------
            # Discover next address
            # ------------------------------------------------

            if tx_to:

                # Ignore zero address
                if tx_to != "0x0000000000000000000000000000000000000000":

                    # Only investigate it if we haven't
                    # already investigated it.
                    if tx_to not in visited_addresses:

                        next_addresses.add(tx_to)


        # Small delay to avoid hammering API
        time.sleep(REQUEST_DELAY)


    # --------------------------------------------------------
    # Move to next hop
    # --------------------------------------------------------

    current_addresses = next_addresses


    print("\nNext hop addresses:", len(current_addresses))


    # Stop if there is nothing else to investigate

    if not current_addresses:

        print("No new addresses found. Stopping.")

        break


# ============================================================
# REMOVE DUPLICATE TRANSACTIONS
# ============================================================

df = pd.DataFrame(all_transactions)


if not df.empty:

    # Same transaction may appear when we investigate
    # multiple addresses.
    df = df.drop_duplicates(
        subset=["hash"]
    )


# ============================================================
# SAVE CSV
# ============================================================

os.makedirs("../data", exist_ok=True)

df.to_csv(
    OUTPUT_PATH,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("COLLECTION COMPLETE")
print("=" * 60)

print(f"Starting address:")
print(START_ADDRESS)

print(f"\nHops explored:")
print(MAX_HOPS)

print(f"\nUnique addresses investigated:")
print(len(visited_addresses))

print(f"\nUnique transactions:")
print(len(df))

print(f"\nSaved to:")
print(OUTPUT_PATH)


HOP 0
Addresses to investigate: 1

Fetching transactions for:
0x846943093f519a47734765bef9ee1136800beb9c
Transactions found: 59

Next hop addresses: 4

HOP 1
Addresses to investigate: 4

Fetching transactions for:
0x0ec72ac1f37f2b6e9182a9d4d3d062594773063c
Transactions found: 1

Fetching transactions for:
0xe7f1725e7734ce288f8367e1bb143e90bb3f0512
Transactions found: 100

Fetching transactions for:
0xd98b9cc1f3a8db4e2a24768e56b240deebac5e34
Transactions found: 2

Fetching transactions for:
0xdb9b1e94b5b69df7e401ddbede43491141047db3
Transactions found: 100

Next hop addresses: 0
No new addresses found. Stopping.

COLLECTION COMPLETE
Starting address:
0x846943093f519A47734765BEF9EE1136800bEb9C

Hops explored:
10

Unique addresses investigated:
5

Unique transactions:
260

Saved to:
../data/transactionA_3hop.csv


### OutWord Flow

In [6]:
import requests
import pandas as pd
import os

API_KEY = "Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

ADDRESS = "0x846943093f519A47734765BEF9EE1136800bEb9C"

url = "https://api.etherscan.io/v2/api"

params = {
    "chainid": "11155111",
    "module": "account",
    "action": "txlist",
    "address": ADDRESS,
    "startblock": 0,
    "endblock": 999999999,
    "page": 1,
    "offset": 100,
    "sort": "asc",
    "apikey": API_KEY
}

response = requests.get(url, params=params)
data = response.json()

if data["status"] == "1":

    # Convert API response to DataFrame
    df = pd.DataFrame(data["result"])

    # --------------------------------------------------
    # KEEP ONLY OUTWARD TRANSACTIONS
    # ADDRESS → OTHER WALLET
    # --------------------------------------------------

    df["from"] = df["from"].str.lower()
    df["to"] = df["to"].str.lower()

    address_lower = ADDRESS.lower()

    df = df[df["from"] == address_lower].copy()

    # --------------------------------------------------
    # KEEP ONLY USEFUL COLUMNS
    # --------------------------------------------------

    df = df[
        [
            "hash",
            "from",
            "to",
            "value",
            "blockNumber",
            "timeStamp",
            "gas",
            "gasPrice"
        ]
    ]

    # --------------------------------------------------
    # CREATE DATA FOLDER
    # --------------------------------------------------

    os.makedirs("../data", exist_ok=True)

    # --------------------------------------------------
    # SAVE CSV
    # --------------------------------------------------

    output_path = "../data/transactionA_outward.csv"

    df.to_csv(output_path, index=False)

    print(f"Saved {len(df)} outward transactions to {output_path}")

else:
    print("API Error:", data)

Saved 51 outward transactions to ../data/transactionA_outward.csv


In [11]:

import requests
import pandas as pd
import os
import time

# ============================================================
# CONFIGURATION
# ============================================================

API_KEY = "Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

START_ADDRESS = "0x846943093f519A47734765BEF9EE1136800bEb9C"

CHAIN_ID = "11155111"       # Sepolia
MAX_HOPS = 5
OFFSET = 100
START_BLOCK = 0
END_BLOCK = 999999999

API_URL = "https://api.etherscan.io/v2/api"

OUTPUT_PATH = "../data/outwordTransactionA_3hop.csv"


# ============================================================
# FUNCTION: GET OUTWARD TRANSACTIONS OF ONE ADDRESS
# ============================================================

def get_outward_transactions(address):

    params = {
        "chainid": CHAIN_ID,
        "module": "account",
        "action": "txlist",
        "address": address,
        "startblock": START_BLOCK,
        "endblock": END_BLOCK,
        "page": 1,
        "offset": OFFSET,
        "sort": "asc",
        "apikey": API_KEY
    }

    try:

        response = requests.get(
            API_URL,
            params=params,
            timeout=20
        )

        response.raise_for_status()

        data = response.json()

    except requests.exceptions.RequestException as e:

        print(f"Request failed for {address}: {e}")
        return []

    except ValueError:

        print(f"Invalid JSON response for {address}")
        return []

    # --------------------------------------------------------
    # Etherscan response handling
    # --------------------------------------------------------

    if data.get("status") != "1":

        message = data.get("message", "Unknown error")

        # "No transactions found" is not really an error
        if "No transactions" in str(message):

            return []

        print(f"API error for {address}: {message}")

        return []

    transactions = data.get("result", [])

    if not isinstance(transactions, list):

        return []

    # --------------------------------------------------------
    # KEEP ONLY OUTWARD TRANSACTIONS
    # address → another address
    # --------------------------------------------------------

    address = address.lower()

    outward = []

    for tx in transactions:

        from_address = tx.get("from", "").lower()
        to_address = tx.get("to", "").lower()

        # Ignore transactions without a destination
        if not to_address:
            continue

        # Only keep:
        #
        # current_address → another_address
        #
        if from_address == address:

            outward.append(tx)

    return outward


# ============================================================
# MULTI-HOP OUTWARD TRACING
# ============================================================

def trace_outward(start_address, max_hops):

    start_address = start_address.lower()

    # --------------------------------------------------------
    # Results from all hops
    # --------------------------------------------------------

    all_transactions = []

    # --------------------------------------------------------
    # Addresses that need to be explored
    #
    # At hop 1:
    #
    # [A]
    #
    # At hop 2:
    #
    # [B, C]
    #
    # At hop 3:
    #
    # [D, E, F]
    # --------------------------------------------------------

    current_addresses = {start_address}

    # --------------------------------------------------------
    # Addresses already queried
    #
    # This prevents:
    #
    # A → B → A
    #
    # from causing infinite recursion.
    # --------------------------------------------------------

    visited_addresses = set()

    # --------------------------------------------------------
    # Process each hop
    # --------------------------------------------------------

    for hop in range(1, max_hops + 1):

        print("\n" + "=" * 60)
        print(f"HOP {hop}")
        print("=" * 60)

        print(f"Addresses to investigate: {len(current_addresses)}")

        next_addresses = set()

        # ----------------------------------------------------
        # Investigate every address discovered at this level
        # ----------------------------------------------------

        for address in current_addresses:

            if address in visited_addresses:

                continue

            print(f"\nChecking: {address}")

            transactions = get_outward_transactions(address)

            # Mark address as processed
            visited_addresses.add(address)

            print(
                f"Found {len(transactions)} outward transactions"
            )

            # ------------------------------------------------
            # Store every outward transaction
            # ------------------------------------------------

            for tx in transactions:

                from_address = tx.get(
                    "from",
                    ""
                ).lower()

                to_address = tx.get(
                    "to",
                    ""
                ).lower()

                # ------------------------------------------------
                # Save transaction + tracing information
                # ------------------------------------------------

                transaction = {

                    "hop": hop,

                    "from": from_address,

                    "to": to_address,

                    "value": tx.get(
                        "value",
                        "0"
                    ),

                    "tx_hash": tx.get(
                        "hash",
                        ""
                    ),

                    "block_number": tx.get(
                        "blockNumber",
                        ""
                    ),

                    "timestamp": tx.get(
                        "timeStamp",
                        ""
                    ),

                    "gas": tx.get(
                        "gas",
                        ""
                    ),

                    "gas_price": tx.get(
                        "gasPrice",
                        ""
                    ),

                    # Address from which this transaction
                    # was discovered
                    "source_address": address

                }

                all_transactions.append(transaction)

                # ------------------------------------------------
                # Add destination to next hop
                # ------------------------------------------------

                if to_address:

                    # Don't immediately go backwards to a wallet
                    # that has already been explored.
                    if to_address not in visited_addresses:

                        next_addresses.add(to_address)

            # ------------------------------------------------
            # Small delay to reduce API pressure
            # ------------------------------------------------

            time.sleep(0.2)

        # ----------------------------------------------------
        # Prepare addresses for next hop
        # ----------------------------------------------------

        current_addresses = next_addresses

        print(
            f"\nNew addresses discovered for next hop: "
            f"{len(current_addresses)}"
        )

        # ----------------------------------------------------
        # If there are no new addresses, tracing is finished
        # ----------------------------------------------------

        if not current_addresses:

            print("\nNo new addresses found. Tracing finished.")

            break

    return all_transactions


# ============================================================
# RUN TRACING
# ============================================================

print("\nStarting outward fund-flow tracing...")

transactions = trace_outward(
    START_ADDRESS,
    MAX_HOPS
)


# ============================================================
# CONVERT TO DATAFRAME
# ============================================================

df = pd.DataFrame(transactions)


# ============================================================
# SAVE RESULTS
# ============================================================

os.makedirs(
    "../data",
    exist_ok=True
)


if not df.empty:

    # Make sure columns are in a useful order

    columns = [

        "hop",
        "from",
        "to",
        "value",
        "tx_hash",
        "block_number",
        "timestamp",
        "gas",
        "gas_price",
        "source_address"

    ]

    df = df[columns]

    df.to_csv(
        OUTPUT_PATH,
        index=False
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("TRACING COMPLETE")
print("=" * 60)

print(f"Starting address: {START_ADDRESS}")

print(f"Maximum hops: {MAX_HOPS}")

print(f"Total transactions found: {len(df)}")

if not df.empty:

    print("\nTransactions by hop:")

    print(
        df.groupby("hop").size()
    )

    print("\nUnique addresses discovered:")

    addresses = set(df["from"]) | set(df["to"])

    print(len(addresses))

    print(f"\nSaved to: {OUTPUT_PATH}")

else:

    print("\nNo outward transactions found.")




Starting outward fund-flow tracing...

HOP 1
Addresses to investigate: 1

Checking: 0x846943093f519a47734765bef9ee1136800beb9c
Found 19 outward transactions

New addresses discovered for next hop: 4

HOP 2
Addresses to investigate: 4

Checking: 0x0ec72ac1f37f2b6e9182a9d4d3d062594773063c
Found 0 outward transactions

Checking: 0xe7f1725e7734ce288f8367e1bb143e90bb3f0512
Found 0 outward transactions

Checking: 0xd98b9cc1f3a8db4e2a24768e56b240deebac5e34
Found 0 outward transactions

Checking: 0xdb9b1e94b5b69df7e401ddbede43491141047db3
Found 0 outward transactions

New addresses discovered for next hop: 0

No new addresses found. Tracing finished.

TRACING COMPLETE
Starting address: 0x846943093f519A47734765BEF9EE1136800bEb9C
Maximum hops: 5
Total transactions found: 19

Transactions by hop:
hop
1    19
dtype: int64

Unique addresses discovered:
5

Saved to: ../data/outwordTransactionA_3hop.csv
